# Tutorial 1: Quickstart

This tutorial walks you through the absolute basics of `rmt-llm-research`:

1. Import the package
2. Compute Marchenko-Pastur bounds
3. Sample a random matrix and compare its spectrum to MP
4. Detect a BBP spike
5. Verify Tracy-Widom fluctuations

**Difficulty:** Beginner
**Estimated time:** 5 minutes

## 1. Import the package

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from rmt_llm.marchenko_pastur import mp_bounds, mp_density, mp_cdf
from rmt_llm.bbp_transition import bbp_lambda_max, bbp_is_spiked
from rmt_llm.tracy_widom import tracy_widom_cdf, tracy_widom_mean

print("Package imported successfully")

## 2. Marchenko-Pastur bounds

For a random $N \times T$ matrix with $q = N/T \leq 1$, the eigenvalue
support of $\frac{1}{T}XX^{\top}$ is $[\lambda_{-}, \lambda_{+}]$ where:

$$\lambda_{\pm} = \sigma^2 (1 \pm \sqrt{q})^2$$

In [ ]:
q, sigma = 0.5, 1.0
lambda_minus, lambda_plus = mp_bounds(q, sigma)
print(f"MP support: [{lambda_minus:.3f}, {lambda_plus:.3f}]")
# Expected: MP support: [0.086, 2.914]

## 3. Compare theory to simulation

Sample a random matrix and compare its eigenvalue density to the MP prediction.

In [ ]:
N, T = 200, 400       # q = N/T = 0.5
X = np.random.randn(N, T) * sigma
eigs = np.linalg.eigvalsh(X @ X.T / T)

# Plot histogram vs MP density
grid = np.linspace(0.01, 3.5, 500)
rho = np.array([mp_density(g, q, sigma) for g in grid])

plt.figure(figsize=(10, 6))
plt.hist(eigs, bins=50, density=True, alpha=0.6, label='Sample')
plt.plot(grid, rho, 'r-', lw=2, label='MP theory')
plt.axvline(lambda_minus, color='k', ls='--', alpha=0.4, label='MP bounds')
plt.axvline(lambda_plus,  color='k', ls='--', alpha=0.4)
plt.xlabel('Eigenvalue')
plt.ylabel('Density')
plt.legend()
plt.title(f'Marchenko-Pastur law (N={N}, T={T}, q={q})')
plt.show()

## 4. Detect a BBP spike

The Baik-Ben Arous-Péché (BBP) transition is a phase transition in the largest
eigenvalue of a spiked random covariance matrix. When the spike strength
$\theta$ exceeds $\theta_c = \sqrt{q}$, the largest eigenvalue pops out of
the MP bulk.

In [ ]:
theta = 1.5      # signal strength
q = 0.5

# Is this spiked?
spiked = bbp_is_spiked(theta, q)
print(f"theta={theta}, q={q} -> spiked: {spiked}")
# spiked: True (because theta > sqrt(q) = 0.707)

# Where does the largest eigenvalue go?
lambda_max = bbp_lambda_max(theta, q, sigma=1.0)
print(f"Predicted lambda_max: {lambda_max:.3f}")
# Predicted lambda_max: 2.729

## 5. Tracy-Widom fluctuations

The largest eigenvalue of a pure (un-spiked) Wishart matrix has fluctuations
of order $N^{-2/3}$ around the MP edge, governed by the Tracy-Widom $F_2$
distribution.

In [ ]:
# Mean of the Tracy-Widom F2 distribution
tw_mean = tracy_widom_mean()
print(f"Tracy-Widom F2 mean: {tw_mean:.4f}")
# Expected: -1.7711

# CDF at the mean
cdf_at_mean = tracy_widom_cdf(tw_mean)
print(f"F2(tw_mean) = {cdf_at_mean:.4f}")

## 6. Verify against the simulated matrix

In [ ]:
# Center and scale the largest eigenvalue
centered = (eigs.max() - lambda_plus) * N**(2/3) / sigma**2
print(f"Centered & scaled lambda_max: {centered:.3f}")
print(f"Tracy-Widom mean:             {tw_mean:.3f}")

# For large N, the centered value should be close to the TW mean.
# For N=200, you'll see substantial finite-size correction — that's
# the Keating-Snaith term (see src/rmt_llm/keating_snaith.py).

## What's next?

- [Tutorial 2: Training TinyGPT](02_training_tinygpt.ipynb)
- [API Reference: marchenko_pastur](https://wild8highlander.github.io/rmt-llm-research/api/rmt-llm/)
- [Theory overview](https://github.com/wild8highlander/rmt-llm-research/blob/main/docs/en/RMT_LLM_Arxiv_Preprint.docx)